In [1]:
import os
import pandas as pd
import numpy as np 
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from transformers import TrainingArguments, Trainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from datasets import Dataset
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
test_batches_file = '/kaggle/input/competitions/problem-3-republic-olymp-tst-homework/test_batches.csv'
test_conclusions_file = '/kaggle/input/competitions/problem-3-republic-olymp-tst-homework/test_conclusions.csv'
test_descriptions_file = '/kaggle/input/competitions/problem-3-republic-olymp-tst-homework/test_descriptions.csv'

In [3]:
test_batches = pd.read_csv(test_batches_file)
test_batches.head()

,batch_id,description_ids,conclusion_ids
0,batch_001,"[""desc_00001"", ""desc_00002"", ""desc_00003"", ""de...","[""conc_00001"", ""conc_00002"", ""conc_00003"", ""co..."
1,batch_002,"[""desc_00011"", ""desc_00012"", ""desc_00013"", ""de...","[""conc_00011"", ""conc_00012"", ""conc_00013"", ""co..."
2,batch_003,"[""desc_00021"", ""desc_00022"", ""desc_00023"", ""de...","[""conc_00021"", ""conc_00022"", ""conc_00023"", ""co..."
3,batch_004,"[""desc_00031"", ""desc_00032"", ""desc_00033"", ""de...","[""conc_00031"", ""conc_00032"", ""conc_00033"", ""co..."
4,batch_005,"[""desc_00041"", ""desc_00042"", ""desc_00043"", ""de...","[""conc_00041"", ""conc_00042"", ""conc_00043"", ""co..."


In [4]:
test_conc = pd.read_csv(test_conclusions_file)
test_conc.head()

,conclusion_id,conclusion
0,conc_00001,КТ-признаков вторичного поражения легких не вы...
1,conc_00002,КТ-данных за органические изменения головного ...
2,conc_00003,"КТ-картина кальцинатов, участков пневмофиброза..."
3,conc_00004,КТ-данных за вторичное поражение лимфатических...
4,conc_00005,КТ-картина состояния после ПХТ по поводу MTS-п...


In [5]:
test_desc = pd.read_csv(test_descriptions_file)
test_desc.head()

,description_id,description
0,desc_00001,На серии КТ-сканов - лимфоузлы малого таза и п...
1,desc_00002,На серии томограмм в 4 сегменте справа и 8 сег...
2,desc_00003,На серии КТ-сканнов – в паренхиме S10 сегмента...
3,desc_00004,На серии КТ-сканов – очаговые изменения плотно...
4,desc_00005,На серии КТ-сканнов – состояние после хирургич...


In [6]:
device = 'cuda'
model_name = "paraphrase-multilingual-MiniLM-L12-v2"

In [7]:
train_df = pd.read_csv('/kaggle/input/competitions/problem-3-republic-olymp-tst-homework/train.csv')
train_df.head()

,description,conclusion
0,На серии томограмм в малом тазу определяется с...,КТ-признаки асцита.
1,На серии КТ-сканов – лимфоузлы малого таза и п...,КТ-данных за вторичное поражение лимфатических...
2,На серии томограмм лимфоузлы малого таза не ув...,КТ-признаков вторичного поражения лимфоузлов и...
3,На серии томограмм лимфоузлы малого таза не ув...,КТ-признаков вторичного поражения лимфоузлов и...
4,На серии томограмм лимфоузлы малого таза не ув...,КТ-признаков вторичного поражения лимфоузлов и...


In [8]:
X_train = train_df['description']
y_train = train_df['conclusion']
len(X_train), len(y_train)

(7373, 7373)

In [9]:
model = SentenceTransformer(model_name).to(device)
loss = MultipleNegativesRankingLoss(model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Получаем backbone модели
backbone = model._first_module().auto_model

for param in backbone.embeddings.parameters():
    param.requires_grad = False

# 2. Замораживаем первые 8 слоев энкодера
for i, layer in enumerate(backbone.encoder.layer):
    if i < 8: 
        for param in layer.parameters():
            param.requires_grad = False

проверка сколько параметров обучается

In [11]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Количество обучаемых параметров: {trainable} Общее: {total} Пропорция: {trainable / total:.4f}')

Количество обучаемых параметров: 7245696 Общее: 117653760 Пропорция: 0.0616


допустим что мы дообучили модель 


In [12]:
train_dataset = Dataset.from_dict({
    'anchor' : list(X_train), 
    'positive' : list(y_train),
})

In [13]:
loss = MultipleNegativesRankingLoss(model)

In [14]:
args = SentenceTransformerTrainingArguments(
    output_dir='my_model',
    num_train_epochs=6,
    per_device_train_batch_size=8,       # ✅ было 32, уменьши до 8
    gradient_accumulation_steps=4,        # ✅ эмулирует batch_size=32
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
    save_strategy='epoch',
    learning_rate=2e-5,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


In [15]:
trainer = SentenceTransformerTrainer(
    model = model, 
    args = args, 
    train_dataset = train_dataset,
    loss = loss
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,2.133997
100,1.328524
150,1.015190
200,0.835926
250,0.769526
300,0.697522
350,0.628546
400,0.632097
450,0.573777
500,0.559886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=696, training_loss=0.8158356978975493, metrics={'train_runtime': 702.3442, 'train_samples_per_second': 62.986, 'train_steps_per_second': 0.991, 'total_flos': 0.0, 'train_loss': 0.8158356978975493, 'epoch': 6.0})

In [16]:
model.save_pretrained('my_finetuned_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
test_desc['embeddings'] = model.encode(test_desc['description'], show_progress_bar=True).tolist()
test_conc['embeddings'] = model.encode(test_conc['conclusion'], show_progress_bar=True).tolist()

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

In [18]:

cur_idx = 0  # начинаем с 0
y_preds = []

for row in range(len(test_batches)):
    desc_ids = np.arange(cur_idx, cur_idx + 10)
    conc_ids = np.arange(cur_idx, cur_idx + 10)

    X_test = np.array(test_desc.iloc[desc_ids]['embeddings'].tolist())
    y_test = np.array(test_conc.iloc[conc_ids]['embeddings'].tolist())

    # Реальные ID заключений этого батча
    actual_conc_ids = test_conc.iloc[conc_ids]['conclusion_id'].values

    knn = NearestNeighbors(n_neighbors=1, metric='cosine')
    knn.fit(y_test)
    indices = knn.kneighbors(X_test, n_neighbors=1, return_distance=False).flatten()

    # ✅ Маппим локальный индекс → реальный conclusion_id
    y_preds.extend(actual_conc_ids[indices])

    cur_idx += 10

print('done')

# Сабмит
sb = pd.DataFrame({
    'description_id': test_desc['description_id'].values,
    'conclusion_id': y_preds,
})
sb.to_csv('sub.csv', index=False)

done


In [19]:
y_preds[:5]

['conc_00004', 'conc_00001', 'conc_00010', 'conc_00002', 'conc_00008']